# Blocco 1: Dal Web al DataFrame
## Python per Excel — Turin Wealth Advisory

---

### Contesto narrativo

Sono le 9:15 di lunedì mattina. Il tuo responsabile, **Dott. Marco Ferretti** (Senior Advisor), si affaccia alla tua scrivania:

> *"Buongiorno. Come sai, Alessandro Bianchi — il figlio 28enne — vuole investire €200.000 in azioni singole. Ha letto di Ferrari, Microsoft, Google... Ha idee molto precise. Prima di incontrarlo venerdì, devo che tu costruisca un'analisi seria su questi titoli. Niente Excel questa volta: useremo Python. Inizia scaricando i dati storici da Yahoo Finance e costruiamo una visione completa."*

Il tuo compito è scaricare, pulire e analizzare i dati storici di **5 titoli azionari**:

| Azienda | Ticker | Mercato | Settore |
|---------|--------|---------|----------|
| Terna SpA | `TRN.MI` | Borsa Italiana | Utilities |
| Ferrari NV | `RACE.MI` | Borsa Italiana | Luxury Automotive |
| Microsoft Corp | `MSFT` | NASDAQ | Technology |
| Alphabet Inc | `GOOGL` | NASDAQ | Technology |
| LVMH | `MC.PA` | Euronext Paris | Luxury Goods |

---

**In questo blocco imparerai a:**
- Scaricare dati finanziari dal web con `yfinance`
- Leggere e navigare un DataFrame pandas
- Confrontare più titoli su un grafico
- Calcolare rendimenti e statistiche di base
- Confrontare dati reali con dati simulati

---

> **Nota tecnica:** Ogni esercizio ha una sezione *DEMO* (già pronta) e una sezione *ESERCIZIO* (da completare tu). Le celle con `# ???` indicano dove devi scrivere il codice. Le **celle soluzione** sono nascoste — prova sempre prima da solo!

In [ ]:
# ============================================================
# SETUP: Librerie e configurazione
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Aggiunge la cartella scripts al path per importare tw_config
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('')), 'scripts'))

try:
    from tw_config import AZIENDE, BENCHMARK, COLORS
    print('tw_config caricato correttamente.')
except ImportError:
    print('tw_config non trovato — uso valori di fallback.')
    AZIENDE = [
        {'nome': 'Terna',     'ticker': 'TRN.MI'},
        {'nome': 'Ferrari',   'ticker': 'RACE.MI'},
        {'nome': 'Microsoft', 'ticker': 'MSFT'},
        {'nome': 'Alphabet',  'ticker': 'GOOGL'},
        {'nome': 'LVMH',      'ticker': 'MC.PA'},
    ]
    BENCHMARK = [
        {'nome': 'S&P 500',      'ticker': '^GSPC'},
        {'nome': 'Euro Stoxx 50','ticker': '^STOXX50E'},
    ]
    COLORS = {}

# -----------------------------------------------------------
# Directory cache per modalità offline
# -----------------------------------------------------------
CACHE_DIR = os.path.join(os.path.dirname(os.path.abspath('')), 'lezione', 'dati_cache')
os.makedirs(CACHE_DIR, exist_ok=True)


def scarica_o_cache(ticker, period='5y'):
    """
    Scarica dati da Yahoo Finance; se il download fallisce,
    carica il file CSV dalla cache locale.

    Parametri
    ---------
    ticker : str   — simbolo del titolo (es. 'MSFT', 'TRN.MI')
    period : str   — periodo storico (es. '5y', '1y', '6mo')

    Ritorna
    -------
    pd.DataFrame con colonne OHLCV
    """
    try:
        df = yf.download(ticker, period=period, progress=False, auto_adjust=True)
        if len(df) > 0:
            # Salva in cache per usi futuri
            nome_file = ticker.replace('^', '').replace('.', '_') + '.csv'
            df.to_csv(os.path.join(CACHE_DIR, nome_file))
            return df
    except Exception as e:
        print(f'  Download fallito per {ticker}: {e}')

    # Fallback su cache locale
    nome_file = ticker.replace('^', '').replace('.', '_') + '.csv'
    path_cache = os.path.join(CACHE_DIR, nome_file)
    if os.path.exists(path_cache):
        print(f'  Uso cache locale per {ticker}')
        return pd.read_csv(path_cache, index_col=0, parse_dates=True)

    raise FileNotFoundError(
        f'Nessun dato disponibile per {ticker}. '
        f'Connettiti a Internet o aggiungi {nome_file} nella cartella dati_cache.'
    )


print()
print('Setup completato!')
print(f'  - Aziende caricate: {len(AZIENDE)}')
print(f'  - Cache directory : {CACHE_DIR}')

### Come funziona `scarica_o_cache()`?

La funzione usa un pattern **try/except** per gestire due scenari:

```
Internet disponibile  →  scarica da Yahoo Finance  →  salva in cache
Internet non disponibile  →  legge il CSV dalla cache locale
```

Questo è un pattern comune in data science: **separation of concerns** tra recupero dati e analisi. Se il download fallisce, il tuo codice di analisi continua a funzionare.

> **Nota su `auto_adjust=True`:** Chiediamo a yfinance di restituire prezzi già aggiustati per split azionari e dividendi. In finanza, quasi sempre lavoriamo con prezzi *adjusted close* per avere serie storiche coerenti.

---
## ES01: Il primo ticker

Ferretti ti guarda mentre apri il terminale:

> *"Inizia con Terna. È un'utility italiana, gestisce la rete di trasmissione elettrica nazionale. Un titolo stabile, difensivo. Scarica i dati degli ultimi 5 anni e dimmi cosa ottieni."*

### Obiettivo
Scaricare dati storici di un titolo e capire la struttura di un DataFrame pandas.

In [ ]:
# ============================================================
# ES01 — DEMO: Scarica Terna (TRN.MI)
# ============================================================

print('Scarico dati Terna (TRN.MI)...')
dati_terna = scarica_o_cache('TRN.MI', period='5y')

print()
print('=== STRUTTURA DEL DATAFRAME ===')
print(f'Dimensioni : {dati_terna.shape}  (righe x colonne)')
print(f'Prima data : {dati_terna.index[0].date()}')
print(f'Ultima data: {dati_terna.index[-1].date()}')
print()
print('Colonne disponibili:')
print(dati_terna.columns.tolist())

print()
print('=== PRIME 5 RIGHE (df.head()) ===')
dati_terna.head()

### Anatomia di un DataFrame pandas

Un DataFrame è come una tabella Excel, ma programmabile:

```
            Open    High     Low   Close   Volume
Date
2020-01-02  6.123   6.200   6.100   6.180  1234567   ← una riga = un giorno di borsa
2020-01-03  6.190   6.220   6.150   6.200   987654
...
↑
Indice (DatetimeIndex) = le date di borsa
```

| Colonna | Significato | Usato per |
|---------|-------------|----------|
| `Open`   | Prezzo apertura | Analisi intraday |
| `High`   | Massimo giornaliero | Range di volatilità |
| `Low`    | Minimo giornaliero | Range di volatilità |
| `Close`  | Prezzo chiusura | **Analisi storica** (il più usato) |
| `Volume` | Azioni scambiate | Conferma segnali |

Per quasi tutte le nostre analisi useremo **solo `Close`**.

**Comandi utili per esplorare un DataFrame:**
```python
df.head(n)     # prime n righe (default: 5)
df.tail(n)     # ultime n righe
df.shape       # (numero righe, numero colonne)
df.columns     # lista colonne
df.index       # indice (date)
df.info()      # tipi di dati e valori nulli
df.describe()  # statistiche descrittive
df['Close']    # seleziona una colonna
```

In [ ]:
# ============================================================
# ES01 — ESERCIZIO: Scarica Ferrari (RACE.MI)
# ============================================================
#
# Il Dott. Ferretti: "Bene. Ora prova tu con Ferrari.
# Voglio vedere le ultime 10 righe dei dati."
#
# Istruzioni:
#   1. Usa scarica_o_cache() per scaricare i dati di Ferrari
#      Il ticker di Ferrari su Borsa Italiana è 'RACE.MI'
#   2. Mostra le ultime 10 righe del DataFrame
#   3. Stampa quanti giorni di borsa hai scaricato
# -----------------------------------------------------------

# 1. Scarica i dati di Ferrari
ticker_ferrari = # ???
dati_ferrari = # ???

# 2. Mostra le ultime 10 righe
# ???

# 3. Stampa il numero di giorni disponibili
# ???

In [ ]:
# ============================================================
# ES01 — SOLUZIONE
# ============================================================

# 1. Scarica i dati di Ferrari
ticker_ferrari = 'RACE.MI'
dati_ferrari = scarica_o_cache(ticker_ferrari, period='5y')

# 2. Mostra le ultime 10 righe
print('=== ULTIME 10 RIGHE — Ferrari (RACE.MI) ===')
display(dati_ferrari.tail(10))

# 3. Numero di giorni di borsa
n_giorni = len(dati_ferrari)
prima_data = dati_ferrari.index[0].date()
ultima_data = dati_ferrari.index[-1].date()
print()
print(f'Giorni di borsa disponibili: {n_giorni}')
print(f'Periodo: dal {prima_data} al {ultima_data}')

# Confronto rapido con Terna
print()
print('Confronto prezzi ultima chiusura:')
print(f"  Terna  (TRN.MI) : €{dati_terna['Close'].iloc[-1]:.2f}")
print(f"  Ferrari (RACE.MI): €{dati_ferrari['Close'].iloc[-1]:.2f}")

---
## ES02: Cinque aziende

Ferretti torna al tuo posto dopo 20 minuti:

> *"Bene. Ma Alessandro non ha parlato solo di Ferrari. Vuole confrontare **cinque titoli**: Terna, Ferrari, Microsoft, Alphabet e LVMH. Hai mai scaricato più ticker contemporaneamente? Con yfinance si può fare in una riga sola. Costruisci una tabella pulita con solo i prezzi di chiusura."*

### Obiettivo
Scaricare più titoli simultaneamente e gestire la struttura **MultiIndex** di pandas.

In [ ]:
# ============================================================
# ES02 — DEMO: Scarica tutti e 5 i titoli
# ============================================================

# Recupera tickers e nomi dalla configurazione
tickers = [az['ticker'] for az in AZIENDE]
nomi    = [az['nome']   for az in AZIENDE]

print(f'Ticker da scaricare: {tickers}')
print('Scarico dati...')

try:
    # yfinance accetta una lista di ticker: scarica tutto in parallelo
    dati_multi = yf.download(tickers, period='5y', progress=False, auto_adjust=True)
    if len(dati_multi) == 0:
        raise ValueError('Dati vuoti')
except Exception as e:
    print(f'Download multiplo fallito: {e}. Uso cache...')
    # Fallback: scarica uno per uno e concatena
    frames = {}
    for az in AZIENDE:
        try:
            df = scarica_o_cache(az['ticker'], period='5y')
            frames[az['ticker']] = df['Close']
        except Exception:
            print(f"  Saltato {az['ticker']}")
    dati_multi = pd.DataFrame(frames)
    prezzi_close = dati_multi
    prezzi_close.columns = nomi

# -----------------------------------------------------------
# Quando si scaricano più ticker, yfinance restituisce
# un DataFrame con MultiIndex sulle colonne:
#
#          Close                           Open  ...
#          GOOGL  MC.PA  MSFT  RACE.MI  TRN.MI
# Date
# 2020-01-02  ...
#
# Dobbiamo estrarre solo il livello 'Close'
# -----------------------------------------------------------

if isinstance(dati_multi.columns, pd.MultiIndex):
    prezzi_close = dati_multi['Close'].copy()
    # Riordina le colonne nell'ordine originale e rinomina
    prezzi_close = prezzi_close[tickers]
    prezzi_close.columns = nomi

# Rimuovi eventuali righe con tutti NaN (festività internazionali)
prezzi_close = prezzi_close.dropna(how='all')

print()
print(f'Dimensioni tabella prezzi: {prezzi_close.shape}')
print()
prezzi_close.head()

### MultiIndex: cosa succede con più ticker?

Quando scarichi più ticker con `yf.download(lista)`, pandas restituisce un DataFrame con **colonne a due livelli** (MultiIndex):

```
Livello 0 (tipo dato): Close  |  Open  |  High  |  Low  |  Volume
Livello 1 (ticker):    MSFT GOOGL ... | MSFT GOOGL ... | ...
```

Per estrarre solo i prezzi di chiusura:
```python
# Seleziona il livello 0 = 'Close'
prezzi_close = dati_multi['Close']
# Risultato: DataFrame normale con una colonna per ticker
```

Attenzione: mercati diversi hanno **calendari di festività diversi**. Il 4 luglio (Independence Day USA) Borsa Italiana è aperta, NYSE è chiuso. Questo genera `NaN` nelle colonne USA.

```python
df.dropna(how='all')  # rimuove solo righe dove TUTTI i valori sono NaN
df.dropna(how='any')  # rimuove righe con ALMENO UN NaN (attenzione!)
df.fillna(method='ffill')  # forward fill: porta avanti l'ultimo prezzo valido
```

In [ ]:
# ============================================================
# ES02 — ESERCIZIO: Analisi del dataset
# ============================================================
#
# Ferretti: "Prima di fare qualsiasi analisi, devo capire
# cosa abbiamo. Dimmi: quanti giorni di borsa copriamo?
# Ci sono dati mancanti? Qual è il range di date comune
# a tutti i titoli?"
#
# Istruzioni:
#   1. Stampa il numero totale di righe e colonne
#   2. Conta i valori mancanti (NaN) per ogni titolo
#      Suggerimento: usa df.isna().sum()
#   3. Trova la prima e ultima data in cui TUTTI i titoli
#      hanno dati (no NaN su nessuna colonna)
#      Suggerimento: usa df.dropna(how='any') poi .index
#   4. Stampa il prezzo di chiusura più recente di ogni titolo
# -----------------------------------------------------------

# 1. Dimensioni del dataset
# ???

# 2. Valori mancanti per titolo
# ???

# 3. Range date con dati completi
prezzi_completi = # ???
data_inizio_comune = # ???
data_fine_comune   = # ???
print(f'Range comune a tutti i titoli: ??? → ???')

# 4. Prezzi più recenti
# ???

In [ ]:
# ============================================================
# ES02 — SOLUZIONE
# ============================================================

# 1. Dimensioni del dataset
print(f'Dimensioni: {prezzi_close.shape[0]} righe × {prezzi_close.shape[1]} colonne')
print()

# 2. Valori mancanti per titolo
nan_per_titolo = prezzi_close.isna().sum()
print('Valori mancanti per titolo:')
for titolo, nan_count in nan_per_titolo.items():
    print(f'  {titolo:12s}: {nan_count} NaN')
print()

# 3. Range date con dati completi su tutti i titoli
prezzi_completi    = prezzi_close.dropna(how='any')
data_inizio_comune = prezzi_completi.index[0].date()
data_fine_comune   = prezzi_completi.index[-1].date()
n_giorni_comuni    = len(prezzi_completi)

print(f'Range comune a tutti i titoli:')
print(f'  Da  : {data_inizio_comune}')
print(f'  A   : {data_fine_comune}')
print(f'  Giorni di borsa: {n_giorni_comuni}')
print()

# 4. Prezzi più recenti
print('Prezzi di chiusura più recenti:')
ultima_riga = prezzi_close.iloc[-1]
for titolo, prezzo in ultima_riga.items():
    valuta = 'USD' if titolo in ['Microsoft', 'Alphabet'] else 'EUR'
    print(f'  {titolo:12s}: {prezzo:8.2f} {valuta}')

---
## ES03: Il benchmark

Ferretti si siede accanto a te:

> *"Ottimo. Ora però ho bisogno di contesto. Se Ferrari ha guadagnato il 40% in 5 anni, è tanto o poco? Dipende dal benchmark. In finanza, **un rendimento si giudica sempre rispetto al mercato di riferimento**. Scarica l'S&P 500 e l'Euro Stoxx 50 e costruiamo un grafico di confronto. Ma attenzione: Ferrari vale €300, Microsoft vale $400, il benchmark è a 4000 punti. Come si confrontano? Si **normalizzano a base 100**.*"

### Obiettivo
Normalizzare serie storiche a base 100 per confrontare asset con prezzi assoluti molto diversi.

In [ ]:
# ============================================================
# ES03 — DEMO: Scarica i benchmark e normalizza
# ============================================================

# Scarica i due benchmark
bench_tickers = ['^GSPC', '^STOXX50E']
bench_nomi    = ['S&P 500', 'Euro Stoxx 50']

print('Scarico benchmark...')
frames_bench = {}
for ticker, nome in zip(bench_tickers, bench_nomi):
    try:
        df_b = scarica_o_cache(ticker, period='5y')
        frames_bench[nome] = df_b['Close']
    except Exception as e:
        print(f'  Errore {ticker}: {e}')

bench = pd.DataFrame(frames_bench)
bench = bench.dropna(how='all')

print(f'Benchmark caricati: {bench.columns.tolist()}')
print()

# -----------------------------------------------------------
# NORMALIZZAZIONE A BASE 100
#
# Idea: se il primo giorno = 100, ogni valore successivo
# mostra la performance rispetto all'inizio.
#
# Formula: valore_norm[t] = valore[t] / valore[0] * 100
#
# Es: S&P 500 parte da 3230 → diventa 100
#     Euro Stoxx 50 parte da 3745 → diventa 100
#     Ora si possono confrontare direttamente!
# -----------------------------------------------------------

# Allinea le date (prendi solo le date comuni ai benchmark)
bench_allineato = bench.dropna(how='any')

# Normalizza: dividi ogni colonna per il primo valore e moltiplica per 100
bench_norm = bench_allineato / bench_allineato.iloc[0] * 100

# Grafico
fig, ax = plt.subplots(figsize=(11, 5))

colori_bench = ['#E74C3C', '#3498DB']  # corallo e blu
for col, colore in zip(bench_norm.columns, colori_bench):
    ax.plot(bench_norm.index, bench_norm[col], label=col, color=colore, linewidth=2)

ax.axhline(y=100, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.set_title('Benchmark normalizzati (base 100)', fontsize=14, fontweight='bold')
ax.set_ylabel('Performance (base 100)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print()
performance_finale = bench_norm.iloc[-1] - 100
print('Performance totale nel periodo:')
for nome, perf in performance_finale.items():
    segno = '+' if perf >= 0 else ''
    print(f'  {nome:15s}: {segno}{perf:.1f}%')

### Perché normalizzare a base 100?

La **normalizzazione a base 100** (o *rebasing*) è una tecnica standard in finanza:

```python
# Formula:
serie_normalizzata = serie / serie.iloc[0] * 100
#                           ↑ primo valore diventa 100
```

**Perché serve?**
- Ferrari vale €300, Microsoft $400, S&P 500 è a 4000 punti
- Non puoi confrontare i prezzi assoluti — non hanno senso insieme
- Dopo la normalizzazione, tutti partono da 100 e si confrontano

**Come si legge il grafico:**
- Base 100 = punto di partenza
- Valore 140 = +40% rispetto all'inizio
- Valore 80 = -20% rispetto all'inizio

**`pd.concat()` — unire DataFrame:**
```python
# Unisce due DataFrame con le stesse colonne, in verticale
df_unito = pd.concat([df1, df2], axis=0)

# Unisce due DataFrame affiancando le colonne
df_unito = pd.concat([df1, df2], axis=1)
```

In [ ]:
# ============================================================
# ES03 — ESERCIZIO: Titoli + benchmark sullo stesso grafico
# ============================================================
#
# Ferretti: "Perfetto. Ora metti INSIEME i 5 titoli e i 2
# benchmark. Così Alessandro vede subito chi ha battuto
# il mercato e chi no."
#
# Istruzioni:
#   1. Usa pd.concat() per unire prezzi_close e bench in un
#      unico DataFrame (axis=1, colonne affiancate)
#      Considera solo le date comuni (dropna how='any')
#   2. Normalizza tutto a base 100
#   3. Traccia il grafico con:
#      - Titoli: linee più sottili
#      - Benchmark: linee tratteggiate più spesse
#      - Linea orizzontale a 100 (base)
#      - Legenda, titolo, griglia
#   4. Stampa la performance totale di ogni titolo ordinata
#      dal migliore al peggiore
# -----------------------------------------------------------

# 1. Unisci prezzi_close e bench
tutto = # ???
tutto = tutto.dropna(how='any')  # solo date con tutti i dati

# 2. Normalizza a base 100
tutto_norm = # ???

# 3. Grafico
fig, ax = plt.subplots(figsize=(12, 6))

# Linee per i 5 titoli
for col in prezzi_close.columns:
    if col in tutto_norm.columns:
        ax.plot(# ???)

# Linee tratteggiate per i benchmark
for col in bench.columns:
    if col in tutto_norm.columns:
        ax.plot(# ???, linestyle='--', linewidth=2)

ax.axhline(y=100, color='gray', linestyle=':', linewidth=1)
ax.set_title(# ???)
ax.set_ylabel('Performance (base 100)')
ax.legend(# ???)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 4. Classifica performance totale
print('\nClassifica performance totale (dal migliore al peggiore):')
# ???

In [ ]:
# ============================================================
# ES03 — SOLUZIONE
# ============================================================

# 1. Unisci prezzi_close e benchmark sullo stesso asse temporale
tutto = pd.concat([prezzi_close, bench], axis=1)
tutto = tutto.dropna(how='any')  # solo date con tutti i dati presenti

print(f'Date comuni a titoli + benchmark: {len(tutto)} giorni di borsa')
print()

# 2. Normalizza a base 100
tutto_norm = tutto / tutto.iloc[0] * 100

# 3. Grafico
palette_titoli    = ['#2C3E50', '#E74C3C', '#3498DB', '#27AE60', '#9B59B6']
palette_benchmark = ['#F39C12', '#E67E22']

fig, ax = plt.subplots(figsize=(13, 6))

# Linee titoli
for col, colore in zip(prezzi_close.columns, palette_titoli):
    if col in tutto_norm.columns:
        ax.plot(tutto_norm.index, tutto_norm[col],
                label=col, color=colore, linewidth=1.5, alpha=0.85)

# Linee benchmark (tratteggiate)
for col, colore in zip(bench.columns, palette_benchmark):
    if col in tutto_norm.columns:
        ax.plot(tutto_norm.index, tutto_norm[col],
                label=col, color=colore, linewidth=2.5,
                linestyle='--', alpha=0.9)

# Linea base 100
ax.axhline(y=100, color='gray', linestyle=':', linewidth=1, label='Base (inizio periodo)')

ax.set_title('Titoli vs Benchmark — Performance normalizzata (base 100)',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Performance (base 100)')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 4. Classifica performance totale
perf_totale = (tutto_norm.iloc[-1] - 100).sort_values(ascending=False)
print('Classifica performance totale (dal migliore al peggiore):')
for i, (nome, perf) in enumerate(perf_totale.items(), 1):
    tipo   = 'BENCHMARK' if nome in bench.columns else 'Titolo'
    segno  = '+' if perf >= 0 else ''
    print(f'  {i}. {nome:15s} ({tipo:9s}): {segno}{perf:.1f}%')

---
## ES04: Rendimenti e statistiche

Alessandro Bianchi chiama in ufficio. Lo senti dalla tua scrivania:

> *"Ferretti, grazie per i grafici. Ma io ho bisogno di **numeri**. Qual è il titolo con il miglior rapporto rischio/rendimento? Qual è la correlazione tra Microsoft e Google? Sono troppo simili per tenerli entrambi?"*

Ferretti ti guarda:

> *"Hai sentito. Calcola rendimenti giornalieri, rendimento annualizzato, volatilità e Sharpe Ratio. Poi costruisci la matrice di correlazione."*

### Obiettivo
Calcolare le principali metriche finanziarie: rendimento, volatilità, Sharpe Ratio e correlazione.

In [ ]:
# ============================================================
# ES04 — DEMO: Rendimenti e statistiche
# ============================================================

# Usa i prezzi di chiusura con date comuni (no NaN)
prezzi = prezzi_close.dropna(how='any')

# -----------------------------------------------------------
# RENDIMENTI GIORNALIERI
#
# pct_change() calcola la variazione percentuale giorno su giorno:
#   r[t] = (P[t] - P[t-1]) / P[t-1]
#
# La prima riga sarà NaN (non c'è il giorno precedente)
# dropna() la rimuove
# -----------------------------------------------------------
rendimenti = prezzi.pct_change().dropna()

print(f'Rendimenti giornalieri: {rendimenti.shape[0]} osservazioni')
print()

# -----------------------------------------------------------
# STATISTICHE ANNUALIZZATE
#
# Un anno ha circa 252 giorni di borsa (non 365!)
# Rendimento annualizzato: media giornaliera × 252
# Volatilità annualizzata: deviazione std giornaliera × sqrt(252)
# Sharpe Ratio: rendimento / volatilità (approssimazione semplice)
# -----------------------------------------------------------
GIORNI_ANNO = 252  # Convenzione mercati finanziari

rend_annualizzato = rendimenti.mean() * GIORNI_ANNO
vol_annualizzata  = rendimenti.std()  * np.sqrt(GIORNI_ANNO)
sharpe_ratio      = rend_annualizzato / vol_annualizzata

# Assembla la tabella statistiche
stats = pd.DataFrame({
    'Rendimento ann.': rend_annualizzato,
    'Volatilita ann.' : vol_annualizzata,
    'Sharpe Ratio'    : sharpe_ratio,
}).round(4)

# Ordina per Sharpe Ratio decrescente (il migliore in cima)
stats_ordinata = stats.sort_values('Sharpe Ratio', ascending=False)

print('=== STATISTICHE ANNUALIZZATE ===')
print()

# Stampa formattata
print(f'{"Titolo":12s}  {"Rendimento":>12s}  {"Volatilita":>12s}  {"Sharpe":>8s}')
print('-' * 52)
for titolo, row in stats_ordinata.iterrows():
    r = row['Rendimento ann.']
    v = row['Volatilita ann.']
    s = row['Sharpe Ratio']
    print(f'{titolo:12s}  {r:>+11.1%}  {v:>11.1%}  {s:>8.2f}')

print()
print('(Sharpe Ratio: maggiore = migliore rapporto rischio/rendimento)')

### Le metriche fondamentali

**Rendimento annualizzato:**
$$R_{ann} = \bar{r}_{giornaliero} \times 252$$

**Volatilità annualizzata** (misura del rischio):
$$\sigma_{ann} = \sigma_{giornaliera} \times \sqrt{252}$$
Il fattore $\sqrt{252}$ deriva dalla proprietà della varianza: $\text{Var}(nX) = n \cdot \text{Var}(X)$, quindi $\sigma(nX) = \sqrt{n} \cdot \sigma(X)$.

**Sharpe Ratio** (rendimento per unità di rischio):
$$SR = \frac{R_{ann} - R_f}{\sigma_{ann}}$$
Dove $R_f$ = tasso privo di rischio. Nella demo lo approssimiamo a 0 per semplicità.

| SR | Interpretazione |
|----|----------------|
| < 0 | Peggio del tasso privo di rischio |
| 0 – 1 | Accettabile |
| 1 – 2 | Buono |
| > 2 | Eccellente |

**Matrice di correlazione:**
```python
corr = df.corr()  # valore tra -1 e +1
# +1 = si muovono perfettamente insieme
#  0 = nessuna relazione lineare
# -1 = si muovono in direzioni opposte (utile per diversificazione!)
```

In [ ]:
# ============================================================
# ES04 — ESERCIZIO: Matrice di correlazione
# ============================================================
#
# Alessandro: "Microsoft e Google sono entrambe tech USA.
# Ha senso tenerle entrambe in portafoglio?"
#
# Istruzioni:
#   1. Calcola la matrice di correlazione dei rendimenti
#      Suggerimento: usa .corr() sul DataFrame dei rendimenti
#   2. Mostra la matrice (df.round(2))
#   3. Trova la coppia di titoli con correlazione MINORE
#      (esclusa la diagonale che è sempre 1.0)
#      Suggerimento: usa np.tril() per ottenere solo il triangolo
#      inferiore, poi trova il minimo
#   4. Stampa un'interpretazione: la coppia meno correlata
#      è quella che offre la migliore diversificazione
# -----------------------------------------------------------

# 1. Matrice di correlazione dei rendimenti giornalieri
corr_matrix = # ???

# 2. Mostra la matrice
print('=== MATRICE DI CORRELAZIONE (rendimenti giornalieri) ===')
# ???

# 3. Trova la coppia meno correlata
# Crea una copia della matrice con NaN sulla diagonale e triangolo superiore
# per non contare la correlazione di ogni titolo con se stesso (che è 1.0)
corr_lower = corr_matrix.where(
    np.tril(np.ones(corr_matrix.shape), k=-1).astype(bool)
)

# Trova posizione del minimo
min_corr = # ???  (usa .min().min() per trovare il valore minimo in tutta la matrice)
pos_min  = # ???  (usa np.argwhere per trovare la posizione, o pandas idxmin)

# 4. Stampa l'interpretazione
print()
print(f'Correlazione minima: ??? tra ??? e ???')
print('→ Questa coppia offre la migliore diversificazione.')
print()
print('Correlazione Microsoft-Alphabet:')
# ???

In [ ]:
# ============================================================
# ES04 — SOLUZIONE
# ============================================================

# 1. Matrice di correlazione dei rendimenti
corr_matrix = rendimenti.corr()

# 2. Visualizza
print('=== MATRICE DI CORRELAZIONE (rendimenti giornalieri) ===')
print()
display(corr_matrix.round(3))
print()

# 3. Triangolo inferiore (esclude diagonale)
corr_lower = corr_matrix.where(
    np.tril(np.ones(corr_matrix.shape), k=-1).astype(bool)
)

# Valore minimo (migliore diversificazione)
min_corr = corr_lower.min().min()

# Trova titoli corrispondenti
min_col = corr_lower.min().idxmin()          # colonna con il minimo
min_row = corr_lower[min_col].idxmin()        # riga corrispondente

# Valore massimo (peggiore diversificazione, più simili)
max_corr = corr_lower.max().max()
max_col  = corr_lower.max().idxmax()
max_row  = corr_lower[max_col].idxmax()

# 4. Interpretazione
print(f'Coppia MENO correlata (migliore diversificazione):')
print(f'  {min_row} & {min_col}: corr = {min_corr:.3f}')
print()
print(f'Coppia PIU\' correlata (scarsa diversificazione):')
print(f'  {max_row} & {max_col}: corr = {max_corr:.3f}')
print()

# Correlazione specifica Microsoft-Alphabet
corr_ms_googl = corr_matrix.loc['Microsoft', 'Alphabet']
print(f'Correlazione Microsoft-Alphabet: {corr_ms_googl:.3f}')
print()
if corr_ms_googl > 0.8:
    print("→ Alta correlazione! Tenerle entrambe riduce poco il rischio.")
    print("→ Suggerimento per Alessandro: considera di scegliere una sola fra le due.")
elif corr_ms_googl > 0.5:
    print("→ Correlazione moderata. Si possono tenere entrambe con buona diversificazione.")
else:
    print("→ Bassa correlazione. Ottima diversificazione!")

---
## ES05: Dati reali vs simulati

Mentre stai per chiudere il laptop, Ferretti si ferma:

> *"Aspetta. Come sai, nei nostri file Excel usiamo dati **simulati** — prezzi generati artificialmente che seguono la struttura reale ma non sono esatti. Va bene per le esercitazioni. Ma mi chiedo: quanto si discostano dalla realtà? Confronta i prezzi di Terna nel file Excel con quelli che hai appena scaricato da Yahoo Finance. Voglio vedere la differenza."*

### Obiettivo
Leggere un file Excel con pandas e confrontare dati simulati con dati reali.

In [ ]:
# ============================================================
# ES05 — DEMO: Carica i dati simulati dal file Excel
# ============================================================

import os

# Percorso del file Excel principale
path_dati = os.path.join(
    os.path.dirname(os.path.abspath('')),
    'output',
    'DATI_Turin_Wealth.xlsx'
)

print(f'Cerco il file: {path_dati}')
print(f'File esiste: {os.path.exists(path_dati)}')
print()

try:
    # Leggi il foglio QUOTAZIONI_AZIONI dal file Excel
    # index_col=0  → la prima colonna è l'indice (le date)
    # parse_dates=True → interpreta l'indice come date
    dati_simulati = pd.read_excel(
        path_dati,
        sheet_name='QUOTAZIONI_AZIONI',
        index_col=0,
        parse_dates=True
    )
    print(f'Dati simulati caricati: {dati_simulati.shape}')
    print(f'Colonne: {dati_simulati.columns.tolist()}')
    print()
    print('Prime 5 righe dati simulati:')
    display(dati_simulati.head())

except FileNotFoundError:
    print('File DATI_Turin_Wealth.xlsx non trovato.')
    print('Esegui prima: cd scripts && python create_dati.py')
    print()
    print('Per questo esercizio, generiamo dati simulati di esempio:')

    # Genera dati simulati di fallback per la demo
    np.random.seed(42)
    date_range = pd.date_range(start='2020-01-01', end='2024-12-31', freq='B')
    prezzi_sim = {}
    prezzi_iniziali = {'TRN.MI': 6.0, 'RACE.MI': 150.0, 'MSFT': 160.0, 'GOOGL': 68.0, 'MC.PA': 400.0}
    for ticker, p0 in prezzi_iniziali.items():
        rend_giorn = np.random.normal(0.0003, 0.015, len(date_range))
        prezzi_sim[ticker] = p0 * np.exp(np.cumsum(rend_giorn))
    dati_simulati = pd.DataFrame(prezzi_sim, index=date_range)
    print(f'Dati simulati di fallback: {dati_simulati.shape}')
    display(dati_simulati.head())

print()
print(f'Dati reali   : {prezzi_close.shape}')
print(f'Dati simulati: {dati_simulati.shape}')

### `pd.read_excel()` — leggere file Excel in pandas

```python
df = pd.read_excel(
    'file.xlsx',
    sheet_name='NomeFoglio',   # nome del foglio (o indice numerico)
    index_col=0,               # prima colonna = indice
    parse_dates=True,          # interpreta date automaticamente
    header=0,                  # prima riga = intestazioni colonne
    skiprows=2,                # salta le prime 2 righe (utile con header decorativi)
)
```

**Confronto tra dati reali e simulati:** I dati simulati seguono la stessa **struttura statistica** (media e volatilità simili) ma i prezzi assoluti e i movimenti giornalieri sono diversi. Questo si verifica calcolando la **correlazione** tra le due serie: se è alta (> 0.9), le serie sono molto simili; se è bassa, i dati simulati si discostano molto dalla realtà.

Per allineare due serie con date diverse:
```python
# Metodo 1: reindex sull'indice comune
serie_comune = serie.reindex(altra_serie.index).dropna()

# Metodo 2: join (solo date presenti in entrambe)
df_join = df1.join(df2, how='inner', lsuffix='_reale', rsuffix='_simulato')
```

In [ ]:
# ============================================================
# ES05 — DEMO: Confronto grafico Terna (reale vs simulato)
# ============================================================

# Individua la colonna Terna nei dati simulati
# (il nome potrebbe essere 'TRN.MI', 'Terna', 'TRN', ecc.)
col_terna_sim = None
for col in dati_simulati.columns:
    if 'TRN' in str(col).upper() or 'TERNA' in str(col).upper():
        col_terna_sim = col
        break

# Se non trovata, usa la prima colonna come proxy
if col_terna_sim is None:
    col_terna_sim = dati_simulati.columns[0]
    print(f'Colonna Terna non trovata, uso: {col_terna_sim}')

# Allinea le date: prendi solo le date presenti in entrambe le serie
terna_reale = prezzi_close['Terna'].dropna()
terna_sim   = dati_simulati[col_terna_sim].dropna()

# Date comuni
date_comuni = terna_reale.index.intersection(terna_sim.index)

if len(date_comuni) == 0:
    print('Nessuna data in comune — normalizza i dati simulati.')
    # Alterna: usa range date comparabile
    data_min = max(terna_reale.index.min(), terna_sim.index.min())
    data_max = min(terna_reale.index.max(), terna_sim.index.max())
    terna_reale = terna_reale[(terna_reale.index >= data_min) & (terna_reale.index <= data_max)]
    terna_sim   = terna_sim  [(terna_sim.index   >= data_min) & (terna_sim.index   <= data_max)]
    # Normalizza a base 100 per confronto
    terna_reale_norm = terna_reale / terna_reale.iloc[0] * 100
    terna_sim_norm   = terna_sim   / terna_sim.iloc[0]   * 100
    usa_normalizzato = True
else:
    terna_reale_norm = terna_reale[date_comuni]
    terna_sim_norm   = terna_sim[date_comuni]
    usa_normalizzato = False

# Grafico overlay
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Grafico 1: Serie temporali
ax1 = axes[0]
label_y = 'Performance (base 100)' if usa_normalizzato else 'Prezzo (€)'

ax1.plot(terna_reale_norm.index, terna_reale_norm.values,
         color='#2C3E50', linewidth=1.5, label='Terna — Reale (Yahoo Finance)', alpha=0.9)
ax1.plot(terna_sim_norm.index, terna_sim_norm.values,
         color='#E74C3C', linewidth=1.5, label='Terna — Simulato (Excel)', alpha=0.7,
         linestyle='--')

ax1.set_title('Terna: prezzi reali vs simulati', fontsize=13, fontweight='bold')
ax1.set_ylabel(label_y)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Grafico 2: Differenza (solo se stessa scala)
ax2 = axes[1]
if not usa_normalizzato:
    differenza = terna_reale_norm - terna_sim_norm
    colori_diff = ['#27AE60' if v >= 0 else '#E74C3C' for v in differenza]
    ax2.bar(differenza.index, differenza.values, color=colori_diff, alpha=0.6, width=1)
    ax2.axhline(y=0, color='black', linewidth=0.8)
    ax2.set_title('Differenza (Reale − Simulato)', fontsize=13)
    ax2.set_ylabel('Differenza prezzo (€)')
else:
    differenza = terna_reale_norm - terna_sim_norm
    ax2.plot(differenza.index, differenza.values, color='#9B59B6', linewidth=1)
    ax2.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
    ax2.set_title('Differenza normalizzata (Reale − Simulato)', fontsize=13)
    ax2.set_ylabel('Differenza (punti base 100)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Correlazione tra le due serie
if not usa_normalizzato:
    corr_terna = terna_reale_norm.corr(terna_sim_norm)
    print(f'Correlazione Terna reale vs simulato: {corr_terna:.4f}')
    if corr_terna > 0.95:
        print('→ Ottima corrispondenza: i dati simulati replicano bene la realtà.')
    elif corr_terna > 0.80:
        print('→ Buona corrispondenza: andamento simile, prezzi assoluti diversi.')
    else:
        print('→ Scarsa corrispondenza: i simulati si discostano significativamente.')

In [ ]:
# ============================================================
# ES05 — ESERCIZIO: Confronto per Ferrari
# ============================================================
#
# Ferretti: "Bene per Terna. Ora fai la stessa analisi
# per Ferrari — è il titolo che interessa di più ad Alessandro."
#
# Istruzioni:
#   1. Individua la colonna Ferrari nei dati simulati
#      (cerca 'RACE' o 'Ferrari' nel nome colonna)
#   2. Allinea le date tra serie reale e simulata
#   3. Normalizza entrambe a base 100
#   4. Traccia un grafico con le due serie
#   5. Calcola la correlazione tra le due serie
#      e stampa un'interpretazione
#
# Bonus: calcola anche l'errore medio assoluto
# (MAE = Mean Absolute Error) in punti base 100
# -----------------------------------------------------------

# 1. Trova la colonna Ferrari nei dati simulati
col_ferrari_sim = None
for col in dati_simulati.columns:
    if # ???
        col_ferrari_sim = col
        break

if col_ferrari_sim is None:
    print('Colonna Ferrari non trovata. Colonne disponibili:', dati_simulati.columns.tolist())
else:
    print(f'Colonna Ferrari nei simulati: {col_ferrari_sim}')

    # 2. Allinea le date
    ferrari_reale = # ???
    ferrari_sim   = # ???

    # Normalizza a base 100 (allinea su intervallo comune)
    data_start = # ???  (prendi il massimo tra le date di inizio delle due serie)
    data_end   = # ???  (prendi il minimo tra le date di fine)

    ferrari_reale_norm = # ???
    ferrari_sim_norm   = # ???

    # 3-4. Grafico
    fig, ax = plt.subplots(figsize=(12, 5))
    # ???
    plt.show()

    # 5. Correlazione e interpretazione
    # ???

    # Bonus: MAE
    # ???

In [ ]:
# ============================================================
# ES05 — SOLUZIONE
# ============================================================

# 1. Trova la colonna Ferrari nei dati simulati
col_ferrari_sim = None
for col in dati_simulati.columns:
    if 'RACE' in str(col).upper() or 'FERRARI' in str(col).upper():
        col_ferrari_sim = col
        break

# Fallback: usa la seconda colonna se disponibile
if col_ferrari_sim is None and len(dati_simulati.columns) > 1:
    col_ferrari_sim = dati_simulati.columns[1]
    print(f'Colonna Ferrari non trovata per nome, uso: {col_ferrari_sim}')

if col_ferrari_sim is None:
    print('Impossibile trovare i dati Ferrari simulati.')
else:
    print(f'Colonna Ferrari trovata: "{col_ferrari_sim}"')

    # 2. Estrai le serie e allinea temporalmente
    ferrari_reale = prezzi_close['Ferrari'].dropna()
    ferrari_sim   = dati_simulati[col_ferrari_sim].dropna()

    # Intervallo comune
    data_start = max(ferrari_reale.index.min(), ferrari_sim.index.min())
    data_end   = min(ferrari_reale.index.max(), ferrari_sim.index.max())

    ferrari_reale_fil = ferrari_reale[
        (ferrari_reale.index >= data_start) & (ferrari_reale.index <= data_end)
    ]
    ferrari_sim_fil = ferrari_sim[
        (ferrari_sim.index >= data_start) & (ferrari_sim.index <= data_end)
    ]

    # Normalizza a base 100 (entrambe partono dallo stesso valore)
    ferrari_reale_norm = ferrari_reale_fil / ferrari_reale_fil.iloc[0] * 100
    ferrari_sim_norm   = ferrari_sim_fil   / ferrari_sim_fil.iloc[0]   * 100

    print(f'Periodo analizzato: {data_start.date()} → {data_end.date()}')
    print(f'Osservazioni reali  : {len(ferrari_reale_norm)}')
    print(f'Osservazioni simulate: {len(ferrari_sim_norm)}')
    print()

    # 3-4. Grafico
    fig, ax = plt.subplots(figsize=(12, 5))

    ax.plot(ferrari_reale_norm.index, ferrari_reale_norm.values,
            color='#E74C3C', linewidth=2, label='Ferrari — Reale (Yahoo Finance)')
    ax.plot(ferrari_sim_norm.index, ferrari_sim_norm.values,
            color='#F39C12', linewidth=1.5, label='Ferrari — Simulato (Excel)',
            linestyle='--', alpha=0.8)

    ax.axhline(y=100, color='gray', linestyle=':', linewidth=1)
    ax.set_title('Ferrari NV: prezzi reali vs simulati (base 100)',
                 fontsize=13, fontweight='bold')
    ax.set_ylabel('Performance (base 100)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 5. Correlazione
    # Allinea su date comuni per il calcolo correlazione
    date_comuni_ferr = ferrari_reale_norm.index.intersection(ferrari_sim_norm.index)

    if len(date_comuni_ferr) > 10:
        corr_ferr = ferrari_reale_norm[date_comuni_ferr].corr(
            ferrari_sim_norm[date_comuni_ferr]
        )
        print(f'Correlazione Ferrari reale vs simulato: {corr_ferr:.4f}')

        if corr_ferr > 0.95:
            print('→ Eccellente: i simulati replicano molto bene la dinamica reale.')
        elif corr_ferr > 0.80:
            print('→ Buona: andamento simile ma con differenze nei prezzi assoluti.')
        elif corr_ferr > 0.50:
            print('→ Moderata: qualche somiglianza ma i simulati si discostano.')
        else:
            print('→ Bassa: i simulati non replicano bene la dinamica reale.')

        # Bonus: MAE (Mean Absolute Error in punti base 100)
        mae = np.abs(
            ferrari_reale_norm[date_comuni_ferr] - ferrari_sim_norm[date_comuni_ferr]
        ).mean()
        print()
        print(f'MAE (errore medio assoluto): {mae:.2f} punti (base 100)')
        print(f'→ In media, i simulati si discostano di {mae:.2f} punti rispetto alla base 100.')
    else:
        print('Troppo poche date in comune per calcolare la correlazione.')

---
## Riepilogo del Blocco 1

Ottimo lavoro! In questo blocco hai imparato a:

| Obiettivo | Strumento |
|-----------|----------|
| Scaricare dati dal web | `yfinance.download()` |
| Gestire errori di rete | Pattern `try/except` con fallback su cache |
| Navigare un DataFrame | `.head()`, `.tail()`, `.shape`, `.info()`, `.describe()` |
| Scaricare più ticker | `yf.download(lista)` + gestione MultiIndex |
| Confrontare serie con scale diverse | Normalizzazione a base 100 |
| Calcolare rendimenti | `.pct_change()` |
| Misurare rischio/rendimento | Rendimento annualizzato, volatilità, Sharpe Ratio |
| Analizzare diversificazione | Matrice di correlazione `.corr()` |
| Leggere file Excel | `pd.read_excel()` |

---

### Anteprima Blocco 2: Analisi fondamentale

Nel **Blocco 2** analizzeremo i **bilanci aziendali** con pandas:
- Calcolo di KPI finanziari: P/E ratio, ROE, EBITDA margin, debt-to-equity
- Confronto fondamentale tra i 5 titoli
- Costruzione di una scoring card per Alessandro
- Visualizzazioni avanzate con matplotlib e seaborn

> *Ferretti, soddisfatto, chiude il laptop: "Bene. Domani mattina alle 9 mi mandi un riepilogo di quello che hai trovato. Alessandro arriva venerdì — voglio essere preparato."*

---

**Riferimenti utili:**
- [Documentazione yfinance](https://pypi.org/project/yfinance/)
- [Documentazione pandas](https://pandas.pydata.org/docs/)
- [Investopedia — Sharpe Ratio](https://www.investopedia.com/terms/s/sharperatio.asp)
- [Investopedia — Correlation](https://www.investopedia.com/terms/c/correlation.asp)